# Washington State GHG Emissions: Who Bears the Burden?

**AD450 Final Project — Daniel Rice, Daniel Merced, Phakphoom, Veer**

## Thesis
The burden of industrial greenhouse gas emissions falls unevenly across Washington State. Rural, lower-income counties bear a disproportionate per-capita emissions load compared to wealthier, more populated urban counties.

## Datasets
1. **WA GHG Reporting Program** — Facility-level emissions reported to the state (2012–2023). Source: [data.wa.gov](https://data.wa.gov/)
2. **US Census County Population Estimates** — Annual county population (2012–2023). Source: [census.gov](https://www.census.gov/programs-surveys/popest/data/data-sets.html)
3. **SAIPE Income & Poverty Estimates** — Median household income and poverty rate by county (2024 snapshot). Source: [census.gov SAIPE](https://www.census.gov/programs-surveys/saipe.html)

## Guiding Questions
1. Which counties emit the most greenhouse gases *per person*, and how does that differ from total emissions?
2. How have emissions changed over time across different sectors?
3. Is there a relationship between a county's median income and its per-capita emissions burden?
4. Which industrial sectors dominate emissions in the counties that bear the heaviest per-capita burden?

---
## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns

In [ ]:
!pip install xlrd

In [ ]:
# Read the GHG raw data
ghg_url = "https://data.wa.gov/api/views/idhm-59de/rows.csv?accessType=DOWNLOAD"
ghg_path = "./data/raw/GHG_Reporting_Program_Publication.csv"

if os.path.exists(ghg_path):
    ghg_raw = pd.read_csv(ghg_path)
else:
    ghg_raw = pd.read_csv(ghg_url)
    ghg_raw.to_csv(ghg_path, index=False)

ghg_raw.head()


In [ ]:
# Census Data Sets

# Data for 2020-2024
raw_census_2020_to_2024 = pd.read_excel("data/raw/co-est2024-pop-53.xlsx", skiprows=3, header=0)

# Data for 2010-2019
raw_census_2010_to_2019 = pd.read_excel("data/raw/co-est2020int-pop-53.xlsx", skiprows=3, header=0)

In [ ]:
# Income Data Sets

raw_income = pd.read_excel("data/raw/est24all.xls", skiprows=3, header=0)

---
## 2. Exploratory Data Analysis (EDA)

Before cleaning or joining anything, we explore the raw GHG dataset to understand its shape, contents, and quirks.

### 2.1 Summarize the data (Daniel Rice)
Use `.info()`, `.shape`, and `.head()` to understand the structure of the GHG dataset — column names, dtypes, non-null counts, and a preview of actual values.

In [ ]:
# TODO (Daniel Rice): Summarize ghg_raw — .shape, .info(), .head()
print(ghg_raw.info())

print(ghg_raw.shape)

print(ghg_raw.head())


### 2.2 Basic statistics on numeric columns (Daniel Merced)
Use `.describe()` on the numeric emissions columns. What is the range of reported emissions? What does the distribution look like (mean vs median)? Are there zeros?

In [ ]:
# TODO (Daniel Merced): .describe() on numeric columns, especially emissions columns
# Note any large differences between mean and median (indicates skew/outliers)

# numeric emissions-related columns
emissions_numeric = ghg_raw.select_dtypes(include="number")

# summary statistics
emissions_numeric.describe().T

### 2.3 Value counts of categorical columns (Phakphoom)
Use `.value_counts()` on Sector, County, and City to understand the categorical distribution. How many unique sectors? Which counties appear most often? Are there any unexpected values (e.g., out-of-state counties)?

In [ ]:
# TODO (Phakphoom): value_counts() for Sector, County, and City
# Flag any counties that are NOT in Washington State
print(raw_census_2010_to_2019[~raw_census_2010_to_2019['Unnamed: 0'].str.contains('Washington')].shape[0])
print(raw_census_2020_to_2024[~raw_census_2020_to_2024['Unnamed: 0'].str.contains('Washington')].shape[0])
print(raw_income[~raw_income['Postal Code'].str.contains('WA')].shape[0])

### 2.4 Histograms of numeric columns (Veer)
Plot histograms of `Reported Emissions (MTCO2e)` and at least one gas breakdown column (e.g., Carbon Dioxide, Methane). Are emissions normally distributed or heavily skewed?

In [ ]:
# TODO (Veer): Histograms of Reported Emissions and at least one gas breakdown column
# Consider using log scale if the distribution is extremely skewed

# Log histogram for Reported Emissions
plt.figure(figsize=(8,5))
plt.hist(
    np.log1p(ghg_raw['Reported Emissions (MTCO2e)'].dropna()),
    bins=30,
    color='darkgreen',
    edgecolor='black'
)
plt.title('Log Distribution of Reported Emissions')
plt.xlabel('log(1 + Reported Emissions)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Log histogram for Carbon Dioxide
plt.figure(figsize=(8,5))
plt.hist(
    np.log1p(ghg_raw['Carbon Dioxide (MTCO2e)'].dropna()),
    bins=30,
    color='darkorange',
    edgecolor='black'
)
plt.title('Log Distribution of Carbon Dioxide Emissions')
plt.xlabel('log(1 + Carbon Dioxide Emissions)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

---
## 3. Data Cleaning & Transformation

The raw data has null values, dtype issues, out-of-state rows, and columns we need to derive. We clean all three datasets in this section.

### 3.1 Correct data dtype issues (Daniel Rice)
- Ensure `Year` is int (not float).
- Parse the `Location` column (lat, lon string) into separate `Latitude` and `Longitude` float columns.
- Fix `Primary NAICS Code` to a clean numeric or string type.
- Also clean the census population dataframes: rename the first column to `County`, strip the leading dot and trailing `, Washington` from county names, drop the state total row, and ensure year columns are int.

In [ ]:
# TODO (Daniel Rice): Fix dtypes in ghg_raw
# Also clean raw_census_2010_to_2019 and raw_census_2020_to_2024:
#   - Rename first col to 'County'
#   - Strip leading '.' and trailing ' County, Washington' from county names
#   - Drop the state total row (first data row = 'Washington')
#   - Keep only year columns we need (2012-2019 from the first file, 2020-2023 from the second)

# === Fix dtypes in ghg_raw ===

# Year is already int64, confirmed via .info() — no fix needed

# Parse Location column ("lat, lon" string) into separate float columns
ghg_raw[['Latitude', 'Longitude']] = (
    ghg_raw['Location']
    .str.split(',', expand=True)
    .astype(float)
)

# Clean NAICS code — drop decimal, convert to nullable Int64
ghg_raw['Primary NAICS Code'] = ghg_raw['Primary NAICS Code'].astype('Int64')


# === Clean census 2010-2019 ===

# Rename first column
raw_census_2010_to_2019 = raw_census_2010_to_2019.rename(columns={raw_census_2010_to_2019.columns[0]: 'County'})

# Drop state total row
raw_census_2010_to_2019 = raw_census_2010_to_2019[raw_census_2010_to_2019['County'] != 'Washington']

# Clean county names: ".Adams County, Washington" -> "Adams"
raw_census_2010_to_2019['County'] = (
    raw_census_2010_to_2019['County']
    .str.lstrip('.')
    .str.replace(' County, Washington', '', regex=False)
)

# Keep only 2012-2019 columns (drop 2010, 2011, April 2020 Census, and the April 2010 base)
year_cols_2010s = [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]
census_2012_2019 = raw_census_2010_to_2019[['County'] + year_cols_2010s].copy()


# === Clean census 2020-2024 ===

raw_census_2020_to_2024 = raw_census_2020_to_2024.rename(columns={raw_census_2020_to_2024.columns[0]: 'County'})
raw_census_2020_to_2024 = raw_census_2020_to_2024[raw_census_2020_to_2024['County'] != 'Washington']

raw_census_2020_to_2024['County'] = (
    raw_census_2020_to_2024['County']
    .str.lstrip('.')
    .str.replace(' County, Washington', '', regex=False)
)

# Keep only 2020-2023 (drop April 2020 base and 2024)
year_cols_2020s = [2020, 2021, 2022, 2023]
census_2020_2023 = raw_census_2020_to_2024[['County'] + year_cols_2020s].copy()

### 3.2 Fill NaN values (Daniel Merced)
The gas breakdown columns (Carbon Dioxide, Methane, Nitrous Oxide, HFCs, PFCs, SF6, Fluorinated-Other) have 67 null rows each. Decide on a fill strategy:
- If the row has a `Reported Emissions` value but missing breakdowns, fill gas columns with 0 (they weren't reported, not necessarily absent).
- The `Jurisdiction` column has ~232 nulls — fill with `'Unknown'` or investigate.
- Document your reasoning.

In [ ]:
# TODO (Daniel Merced): Fill NaN values in emissions breakdown columns and Jurisdiction
# Print null counts before and after to show the work

gas_cols = [
    "Biogenic Carbon Dioxide (MTCO2e)",
    "Carbon Dioxide (MTCO2e)",
    "Methane (MTCO2e)",
    "Nitrous Oxide (MTCO2e)",
    "HFCs (MTCO2e)",
    "PFCs (MTCO2e)",
    "Sulfur Hexafluoride (MTCO2e)",
    "Fluorinated - Other (MTCO2e)"
]

reported_col = "Reported Emissions (MTCO2e)"
jurisdiction_col = "Jurisdiction"

# null counts before
print("Null counts before fill:\n")
print(ghg_raw[gas_cols + [jurisdiction_col]].isna().sum())

# fill gas breakdowns with 0 when total reported emissions exists
has_reported = ghg_raw[reported_col].notna()
ghg_raw.loc[has_reported, gas_cols] = ghg_raw.loc[has_reported, gas_cols].fillna(0)

# fill missing jurisdiction with Unknown
ghg_raw[jurisdiction_col] = ghg_raw[jurisdiction_col].fillna("Unknown")

# null counts after
print("\nNull counts after fill:\n")
print(ghg_raw[gas_cols + [jurisdiction_col]].isna().sum())

# 64 rows had missing gas breakdowns but did have a 
# total reported emissions value, so they were filled with 0
# 3 rows had missing gas breakdowns and also did not have a 
# reported emissions value, so they stayed missing

### 3.3 Remove inaccurate / out-of-scope data (Phakphoom)
The County value_counts revealed non-WA counties (e.g., `Greater Vancouver`, `Umatilla`, `Sonoma`, `Los Angeles`). These are facilities that report to WA but are physically located elsewhere.
- Identify and remove rows where County is not a valid WA county.
- Also remove the `Supplier` sector rows — these represent fuel suppliers, not physical emitters in a location, and will distort per-capita analysis.
- Document how many rows are removed and why.

In [ ]:
# WA counties reference
wa_counties = [
    "Adams", "Asotin", "Benton", "Chelan", "Clallam", "Clark", "Columbia",
    "Cowlitz", "Douglas", "Ferry", "Franklin", "Garfield", "Grant",
    "Grays Harbor", "Island", "Jefferson", "King", "Kitsap", "Kittitas",
    "Klickitat", "Lewis", "Lincoln", "Mason", "Okanogan", "Pacific",
    "Pend Oreille", "Pierce", "San Juan", "Skagit", "Skamania",
    "Snohomish", "Spokane", "Stevens", "Thurston", "Wahkiakum",
    "Walla Walla", "Whatcom", "Whitman", "Yakima"
]

# Track initial size of the dataset before cleaning
raw_count_before_cleaning = len(ghg_raw)

# Normalize County names: strip whitespace, remove " County, Washington" and " County" suffixes
ghg_raw["County"] = (
    ghg_raw["County"]
    .astype(str)
    .str.strip()
    .str.replace(" County, Washington", "", regex=False)
    .str.replace(" County", "", regex=False)
)

# Build masks to identify rows to remove
invalid_county_mask = ~ghg_raw["County"].isin(wa_counties)
supplier_mask = ghg_raw["Sector"].eq("Supplier")

# Diagnostics before removal
out_of_state_counties = (
    ghg_raw.loc[invalid_county_mask, "County"]
    .value_counts()
)

rows_removed_invalid = invalid_county_mask.sum()
rows_removed_supplier = supplier_mask.sum()

# Avoid double counting by taking the union of the two masks
total_rows_removed = (invalid_county_mask | supplier_mask).sum()

# Apply fitering for both conditions in a single step to avoid multiple passes and potential double counting
ghg_raw = ghg_raw.loc[
    ~(invalid_county_mask | supplier_mask)
].copy()

# Drop unnecessary columns that won't be used in analysis (Notes, Latitude, Longitude)
ghg_raw = ghg_raw.drop(
    columns=["Notes", "Latitude", "Longitude"],
    errors="ignore"
)

# Count after cleaning
raw_count_after_cleaning = len(ghg_raw)

# Sanity check
assert raw_count_before_cleaning - raw_count_after_cleaning == total_rows_removed, \
    "Mismatch in removal counts!"

# Create summary table of removed rows
summary_of_removed_rows = pd.DataFrame(
    {
        "Rows Removed": [
            rows_removed_invalid,
            rows_removed_supplier,
            total_rows_removed
        ]
    },
    index=[
        "Invalid Counties",
        "Supplier Sector",
        "Total (Unique Rows Removed)"
    ]
)

print("Row count before cleaning:", raw_count_before_cleaning)
print("Row count after cleaning:", raw_count_after_cleaning)

display(summary_of_removed_rows.head())
display(out_of_state_counties.head(20))

# Clean income data
raw_income_copy = raw_income.copy()

# Count before cleaning
row_count_before_filering_income = len(raw_income_copy)

# Mask for non-WA entries
wa_income_mask = raw_income_copy["Postal Code"].eq("WA")

non_wa_postal_codes = (
    raw_income_copy.loc[~wa_income_mask, "Postal Code"]
    .value_counts()
)

# Count how many non-WA entries
non_wa_count_income = (~wa_income_mask).sum()

# Filter to keep only WA entries
income_clean = raw_income_copy.loc[wa_income_mask].copy()

display(income_clean.loc[income_clean["Name"] == "Washington"])

# Standardize county names for joining
income_clean["County"] = (
        income_clean["Name"]
        .astype(str)
        .str.replace(r"\s+County$", "", regex=True)
        .str.strip()
)

# Keep only county level columns needed for analysis
income_clean = income_clean[[
    "County",
    "Median Household Income",
    "Poverty Percent, All Ages",
]]

# Count after cleaning
row_count_after_filering_income = len(income_clean)

# Sanity check
assert row_count_before_filering_income - row_count_after_filering_income == non_wa_count_income, \
    "Mismatch in removal counts for income data!"

# Create summary table for income data cleaning
summary_income_cleaning = pd.DataFrame(
    {
        "Rows Removed": [non_wa_count_income],
        "Rows Remaining": [row_count_after_filering_income]
    },
    index=["Non-WA Postal Codes"]
)

print("Income Data - Row count before cleaning:", row_count_before_filering_income)
print("Income Data - Row count after cleaning:", row_count_after_filering_income)

display(summary_income_cleaning.head())
display(non_wa_postal_codes.head(20))
display(income_clean.head())
income_clean["County"].nunique()
sorted(income_clean["County"].unique())

### 3.4 Add derivative columns (Veer)
Create new columns that will power our analysis:
- `Non_CO2_Emissions`: sum of Methane + Nitrous Oxide + HFCs + PFCs + SF6 + Fluorinated-Other (shows how much of each facility's footprint is non-CO2 gases)
- Clean the income data: filter `raw_income` to WA counties only (`State FIPS Code == 53`, `County FIPS Code != 0`), strip `' County'` from the `Name` column, keep only `Name`, `Poverty Percent, All Ages`, and `Median Household Income`.

In [ ]:
# TODO (Veer): Add Non_CO2_Emissions column to ghg_raw
# Also clean raw_income into a tidy income_df with columns: County, Poverty_Pct, Median_Income

# Add Non_CO2_Emissions column to ghg_raw
ghg_raw['Non_CO2_Emissions'] = ghg_raw[
    [
        'Methane (MTCO2e)',
        'Nitrous Oxide (MTCO2e)',
        'HFCs (MTCO2e)',
        'PFCs (MTCO2e)',
        'Sulfur Hexafluoride (MTCO2e)',
        'Fluorinated - Other (MTCO2e)'
    ]
].sum(axis=1)

# Clean raw_income into income_df
income_df = raw_income[
    (raw_income['State FIPS Code'] == 53) &
    (raw_income['County FIPS Code'] != 0)
].copy()

income_df['Name'] = income_df['Name'].str.replace(' County', '', regex=False)

income_df = income_df[
    ['Name', 'Poverty Percent, All Ages', 'Median Household Income']
]

income_df = income_df.rename(columns={
    'Name': 'County',
    'Poverty Percent, All Ages': 'Poverty_Pct',
    'Median Household Income': 'Median_Income'
})

ghg_raw[['Reported Emissions (MTCO2e)', 'Carbon Dioxide (MTCO2e)', 'Non_CO2_Emissions']].head()


---
## 4. Data Joining

We now combine our three datasets to enable per-capita and income-based analysis.

### 4.1 Concatenate census population dataframes (Daniel Rice)
Melt each cleaned census dataframe from wide to long format (columns: `County`, `Year`, `Population`), then concatenate them into a single `population_df` covering 2012–2023.
- Watch for the 2020 overlap between the two files — pick one.
- Verify shape: should be 39 counties × 12 years = 468 rows.

In [ ]:
# Daniel Rice

# Melt 2012-2019 from wide to long
pop_2012_2019 = census_2012_2019.melt(
    id_vars='County',
    var_name='Year',
    value_name='Population'
)

# Melt 2020-2023 from wide to long
pop_2020_2023 = census_2020_2023.melt(
    id_vars='County',
    var_name='Year',
    value_name='Population'
)

# Ensure Year is int (melt can produce float from Excel headers)
pop_2012_2019['Year'] = pop_2012_2019['Year'].astype(int)
pop_2020_2023['Year'] = pop_2020_2023['Year'].astype(int)

# Concatenate into one population dataframe
population_df = pd.concat([pop_2012_2019, pop_2020_2023], ignore_index=True)

# Verify: 39 counties × 12 years = 468 rows
print("Shape:", population_df.shape)
print(population_df.head())
print(population_df.tail(15))

### 4.2 Merge GHG data with population on County + Year (Daniel Merced)
Use `pd.merge()` to join the cleaned GHG data with `population_df` on `County` and `Year`. This gives every emissions row a population context.
- Use a left merge to keep all GHG rows.
- Check how many rows have null Population after the merge (counties that don't match).

In [ ]:
# (Daniel Merced)
# Merge GHG data with population data on County + Year
ghg_with_population = pd.merge(
    ghg_raw,
    population_df,
    on=["County", "Year"],
    how="left"
)

# Check how many rows did not find a population match
null_population_count = ghg_with_population["Population"].isna().sum()

print("Merged shape:", ghg_with_population.shape)
print("Rows with null Population:", null_population_count)

### 4.3 Merge with income data on County (Phakphoom)
Use `pd.merge()` to join the income/poverty data onto the GHG+population dataframe on `County`.
- This is a many-to-one join (many GHG rows per county, one income row per county).
- Verify the join worked by spot-checking a known county like King or Lewis.

In [ ]:
# Spot-check a couple of counties

# Merge GHG + Population with Income data on County
ghg_population_income = pd.merge(
    ghg_with_population,
    income_clean,
    on="County",
    how="left",
    validate="many_to_one"
)

# Convert income columns to numeric, coercing errors to NaN (should be none after cleaning)
ghg_population_income["Median Household Income"] = pd.to_numeric(
    ghg_population_income["Median Household Income"], errors="coerce"
)

ghg_population_income["Poverty Percent, All Ages"] = pd.to_numeric(
    ghg_population_income["Poverty Percent, All Ages"], errors="coerce"
)

# Merge diagnostics
null_median_income_count = ghg_population_income["Median Household Income"].isna().sum()
null_poverty_pct_count = ghg_population_income["Poverty Percent, All Ages"].isna().sum()

print("Merged shape with income:", ghg_population_income.shape)
print("Rows with null Median Household Income:", null_median_income_count)
print("Rows with null Poverty Percent, All Ages:", null_poverty_pct_count)

# Check for any counties that didn't match
unmatched_counties = (
    ghg_population_income.loc[
        ghg_population_income["Median Household Income"].isna() &
        ghg_population_income["Poverty Percent, All Ages"].isna(),
        "County"
    ]
    .drop_duplicates()
    .sort_values()
)

display(unmatched_counties)

# Sanity check: all unmatched counties should be in the original GHG data but not in the income data

# Merge should not change row count
assert len(ghg_population_income) == len(ghg_with_population), \
    "Row count changed after merging income data"

# No missing income fields after merge
assert ghg_population_income["Median Household Income"].isna().sum() == 0, \
    "There are still missing Median Household Income values after merge"

assert ghg_population_income["Poverty Percent, All Ages"].isna().sum() == 0, \
    "Some rows are missing Poverty Percent, All Ages after merge."

# Each county should map to exactly one income value
income_consistency_check = ghg_population_income.groupby("County")[
    ["Median Household Income", "Poverty Percent, All Ages"]
].nunique()

assert (income_consistency_check["Median Household Income"] == 1).all(), \
    "A county has multiple Median Household Income values."

assert (income_consistency_check["Poverty Percent, All Ages"] == 1).all(), \
    "A county has multiple Poverty Percent values."

# exact duplicate check
exact_duplicate_rows = ghg_population_income.duplicated().sum()
print("Exact duplicate rows:", exact_duplicate_rows)

# each county should map to one income / poverty value
income_consistency_check = ghg_population_income.groupby("County")[
    ["Median Household Income", "Poverty Percent, All Ages"]
].nunique()

assert (income_consistency_check["Median Household Income"] == 1).all()
assert (income_consistency_check["Poverty Percent, All Ages"] == 1).all()

# counties present in income but not in final facility-level table
missing_counties_from_final = sorted(
    set(income_clean["County"]) - set(ghg_population_income["County"])
)
print("WA counties not represented in final GHG table:", missing_counties_from_final)

print("sanity checks passed.")

# Final spot-check of merged data
print("Shape:", ghg_population_income.shape)
display(ghg_population_income.head())
display(ghg_population_income.sample(5, random_state=42))

# Display summary info and null counts for the final merged dataset
display(ghg_population_income.info())

# Check for any remaining null values in the final dataset
null_summary = ghg_population_income.isna().sum().sort_values(ascending=False)
display(null_summary[null_summary > 0])

# Summary statistics for key columns in the final dataset
key_columns = [
    "County",
    "Year",
    "Population",
    "Median Household Income",
    "Poverty Percent, All Ages"
]

display(ghg_population_income[key_columns].describe(include="all"))

# Check the number of unique income values per county to confirm consistency
county_income_variation = ghg_population_income.groupby("County")[
    ["Median Household Income", "Poverty Percent, All Ages"]
].nunique()

display(county_income_variation.sort_values("Median Household Income", ascending=False).head())

year_coverage = ghg_population_income.groupby("County")["Year"].nunique()

display(year_coverage.describe())
display(year_coverage.sort_values().head())

ghg_population_income.loc[
    ghg_population_income["County"].isin(["King", "Yakima", "Spokane"]),
    ["County", "Year", "Population", "Median Household Income", "Poverty Percent, All Ages"]
].sort_values(["County", "Year"]).head(20)

duplicates = ghg_population_income.duplicated(
    subset=["County", "Year", "Sector"], keep=False
)

print("Duplicate rows:", duplicates.sum())


### 4.4 Join aggregated dataframes on index (Veer)
To demonstrate index-based joining:
- Create a df of total emissions per county (all years summed), indexed by County.
- Create a df of average population per county (across all years), indexed by County.
- Use `.join()` to combine them on the County index.
- Add a `Per_Capita_Emissions` column (total emissions / avg population).

In [ ]:
# TODO (Veer): Create two indexed dfs and .join() them, then compute Per_Capita_Emissions
# This df will be key for the final visualizations

# Keep only WA counties that appear in the population data
ghg_wa = ghg_raw[ghg_raw['County'].isin(population_df['County'])].copy()

# Total emissions per county (all years summed), indexed by County
emissions_by_county = (
    ghg_wa
    .groupby('County')['Reported Emissions (MTCO2e)']
    .sum()
    .to_frame(name='Total_Emissions')
)

# Average population per county (across all years), indexed by County
avg_population_by_county = (
    population_df
    .groupby('County')['Population']
    .mean()
    .to_frame(name='Avg_Population')
)

# Join on County index
county_emissions = emissions_by_county.join(avg_population_by_county)

# Add per-capita emissions column
county_emissions['Per_Capita_Emissions'] = (
    county_emissions['Total_Emissions'] / county_emissions['Avg_Population']
)

county_emissions.sort_values('Per_Capita_Emissions', ascending=False).head(10)

---
## 5. Aggregation & Grouping Operations

### 5.1 Aggregation on all data (Daniel Rice)
Compute total reported emissions across all of WA for each year (2012–2023). Is the state's total emissions trending up or down?

In [ ]:
# (Daniel Rice): Total reported emissions across all of WA, by year
annual_emissions = (
    ghg_wa
    .groupby('Year')['Reported Emissions (MTCO2e)']
    .sum()
    .reset_index()
    .rename(columns={'Reported Emissions (MTCO2e)': 'Total_Emissions'})
)

# Format for readability
annual_emissions['Total_Emissions_M'] = (annual_emissions['Total_Emissions'] / 1e6).round(2)

print("Total WA Industrial Emissions by Year (millions MTCO2e):\n")
print(annual_emissions[['Year', 'Total_Emissions_M']].to_string(index=False))

# Quick trend check
first = annual_emissions.iloc[0]['Total_Emissions']
last = annual_emissions.iloc[-1]['Total_Emissions']
pct_change = ((last - first) / first * 100).round(1)
print(f"\nChange from {annual_emissions.iloc[0]['Year']} to {annual_emissions.iloc[-1]['Year']}: {pct_change}%")

### 5.2 Aggregation on a groupby (Daniel Merced)
Group by `Sector` and `Year`, then sum emissions. This reveals which industries are growing or shrinking their footprint over time.

In [ ]:
# Display the top 5 sectors' trends

# Group by Sector and Year, then sum reported emissions
sector_year_emissions = (
    ghg_wa
    .groupby(["Sector", "Year"])["Reported Emissions (MTCO2e)"]
    .sum()
    .reset_index()
)

# Find the top 5 sectors by total emissions across the years
top_5_sectors = (
    sector_year_emissions
    .groupby("Sector")["Reported Emissions (MTCO2e)"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index
)

# Filter only those top 5 sectors
top_5_sector_trends = sector_year_emissions[
    sector_year_emissions["Sector"].isin(top_5_sectors)
].copy()

# Display the grouped data
print("Top 5 sectors by total emissions across all years:")
print(top_5_sectors.tolist())

display(
    top_5_sector_trends
    .sort_values(["Sector", "Year"])
)

### 5.3 Pivot table — emissions by county and year (Phakphoom)
Create a pivot table with counties as rows, years as columns, and total reported emissions as values. This gives a complete emissions matrix for WA.

In [ ]:
# Show the top 10 counties

# Pivot table of total emissions by County and Year
emissions_pivot = pd.pivot_table(
    ghg_wa,
    index='County',
    columns='Year',
    values='Reported Emissions (MTCO2e)',
    aggfunc='sum',
    fill_value=0
)

# Rank counties by total emissions across all years
county_total_emissions = emissions_pivot.sum(axis=1).sort_values(ascending=False)

# Keep only the top 10 counties
top_10_counties = emissions_pivot.loc[county_total_emissions.head(10).index]

# Sanity check
print("Pivot table shape:", emissions_pivot.shape)
print("Years covered in pivot:", emissions_pivot.columns.tolist())

# Display the top 10 counties' emissions by year
print("Top 10 Counties' Emissions by Year:")
display(top_10_counties)

### 5.4 Cross-tabulation — sector presence by county (Veer)
Create a cross-tabulation showing how many reporting facilities exist for each sector in each county. Which counties have the most diverse industrial base? Which are dominated by a single sector?

In [ ]:
# TODO (Veer): pd.crosstab() — County vs Sector, showing facility counts
# Filter to top 10 counties for readability

# Use WA-only data to avoid out-of-state counties
ghg_wa = ghg_raw[ghg_raw['County'].isin(population_df['County'])].copy()

# Create cross-tabulation: County vs Sector (facility counts)
county_sector_crosstab = pd.crosstab(ghg_wa['County'], ghg_wa['Sector'])

# Get top 10 counties by total number of facilities
top10 = county_sector_crosstab.sum(axis=1).sort_values(ascending=False).head(10).index

# Filter to top 10 counties
county_sector_top10 = county_sector_crosstab.loc[top10]

county_sector_top10

---
## 6. Data Visualization & Analysis

We now answer our guiding questions with polished visualizations.

### Q1: Which counties emit the most per person, and how does that differ from total emissions? (Daniel Rice)

Create a side-by-side bar chart (or two ranked bar charts) comparing:
- Top 10 counties by **total** emissions
- Top 10 counties by **per-capita** emissions

The key finding should be visible: counties like King lead in total emissions but are low per-capita, while rural/industrial counties like Lewis or Cowlitz dominate per-capita.

In [ ]:
# (Daniel Rice): Side-by-side comparison of total vs per-capita emissions

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Top 10 by TOTAL emissions
top_total = county_emissions.sort_values('Total_Emissions', ascending=False).head(10)
axes[0].barh(
    top_total.index[::-1],
    top_total['Total_Emissions'].values[::-1] / 1e6,
    color='#5A6A7E'
)
axes[0].set_xlabel('Total Emissions (Millions MTCO₂e)', fontsize=11)
axes[0].set_title('Top 10 Counties by Total Emissions', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='y', labelsize=10)

# Right: Top 10 by PER-CAPITA emissions
top_percap = county_emissions.sort_values('Per_Capita_Emissions', ascending=False).head(10)
axes[1].barh(
    top_percap.index[::-1],
    top_percap['Per_Capita_Emissions'].values[::-1],
    color='#C05621'
)
axes[1].set_xlabel('Per-Capita Emissions (MTCO₂e per person)', fontsize=11)
axes[1].set_title('Top 10 Counties by Per-Capita Emissions', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='y', labelsize=10)

fig.suptitle('The Rankings Flip When You Account for Population', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('q1_total_vs_percapita.png', dpi=150, bbox_inches='tight')
plt.show()


King County leads in total emissions but, with 2.4 million residents, its per-capita burden is modest. When we normalize by population, rural industrial counties like Lewis, Cowlitz, and Klickitat surge to the top. These are small communities (often under 100,000 people) with a handful of heavy-emitting facilities. The "who pollutes the most" question has a completely different answer depending on whether you're counting tons or tons-per-person.

### Q2: How have emissions changed over time by sector? (Daniel Merced)

Create a line chart showing total emissions per year for the top 5 sectors. Are any sectors declining? Did COVID (2020) have a visible impact? Annotate or highlight any notable trends.

In [ ]:
# Include legend, axis labels, title, and any annotations

plt.figure(figsize=(12, 7))

# Plot one line per top sector
for sector in top_5_sectors:
    sector_data = top_5_sector_trends[top_5_sector_trends["Sector"] == sector]
    plt.plot(
        sector_data["Year"],
        sector_data["Reported Emissions (MTCO2e)"],
        marker="o",
        label=sector
    )

plt.title("Top 5 Sectors by Reported Emissions Over Time")
plt.xlabel("Year")
plt.ylabel("Total Reported Emissions (MTCO2e)")
plt.legend(title="Sector")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show() 

These are the top 5 sectors by reported emissions. Our top reported emissions contributor is the power sector, where despite the largest fluctuations over time has consistently maintained it's position above the other four. Pulp & Paper and Petrol are stable and are similar in reported emissions, the same is true for metals and wood products.

### Q3: Is there a relationship between county income and per-capita emissions? (Phakphoom)

Create a scatter plot with:
- X-axis: Median Household Income
- Y-axis: Per-capita emissions
- Optional: size or color by population

Label notable outlier counties. Does lower income correlate with higher emissions burden?

In [ ]:
# Label key outlier counties, add title and axis labels

# Merge count level emissions with income data for scatter plot
county_emissions_by_income = county_emissions.join(
    ghg_population_income[['Median Household Income', "County"]]
    .drop_duplicates(subset="County")
    .set_index("County")
)

# Median Household Income vs Per-Capita Emissions scatter plot
plt.figure(figsize=(10, 6))

sns.regplot(
    data=county_emissions_by_income,
    x="Median Household Income",
    y="Per_Capita_Emissions",
    scatter_kws={"s": 60, "alpha": 0.7},
    line_kws={"color": "red"}
)

# Labeling top 3 outliers
top_outliers = county_emissions_by_income.sort_values(
    'Per_Capita_Emissions', ascending=False
).head(5)

for county, row in top_outliers.iterrows():
    plt.text(
        row['Median Household Income'],
        row['Per_Capita_Emissions'],
        county,
        fontsize=9,
        ha='right'
    )

# Labels and title
plt.title("Income vs Per-Capita Emissions (with Trend Line)")
plt.xlabel("Median Household Income")
plt.ylabel("Per Capita Emissions")

plt.grid(True)
plt.show()

corr = county_emissions_by_income["Median Household Income"].corr(
    county_emissions_by_income["Per_Capita_Emissions"]
)
print(f"The correlation between Median Household Income and Per-Capita Emissions is {corr}.")

The results suggest that per-capita emissions are not strongly associated with county income levels. The weak negative correlation indicates that income alone does not explain emissions patterns. Instead, emissions appear to be driven by the presence of large industrial facilities, which are unevenly distributed across counties. This leads to extreme per-capita values in certain lower-population counties, highlighting that structural and geographic factors play a more significant role than socioeconomic characteristics.

### Q4: Which sectors dominate emissions in the highest per-capita counties? (Veer)

Take the top 5 counties by per-capita emissions. Create a stacked bar chart showing the sector breakdown of emissions in each county. Are these counties dominated by one industry, or is the burden spread across sectors?

In [ ]:
# TODO (Veer): Stacked bar chart — sector breakdown for top 5 per-capita counties
# Clear legend, title, and axis labels

# Get top 5 counties by per-capita emissions
top5 = county_emissions.sort_values('Per_Capita_Emissions', ascending=False).head(5).index

# Filter WA-only data
ghg_wa = ghg_raw[ghg_raw['County'].isin(population_df['County'])].copy()

# Group by County + Sector and sum emissions
sector_emissions = (
    ghg_wa[ghg_wa['County'].isin(top5)]
    .groupby(['County', 'Sector'])['Reported Emissions (MTCO2e)']
    .sum()
    .unstack(fill_value=0)
)

# Plot stacked bar chart
sector_emissions.plot(
    kind='bar',
    stacked=True,
    figsize=(10,6)
)

plt.title('Sector Breakdown of Emissions in Top 5 Per-Capita Counties')
plt.xlabel('County')
plt.ylabel('Total Emissions (MTCO2e)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Sector')
plt.tight_layout()
plt.savefig('sector_breakdown_of_emissions_top5_percapita.png', dpi=150, bbox_inches='tight')
plt.show()

The stacked bar chart reveals that emissions in the highest per-capita counties are largely driven by a small number of dominant industries rather than a balanced mix of sectors. For example, counties such as Cowlitz and Lewis are heavily influenced by industries like pulp and paper, power plants, and natural gas systems, which account for the majority of their emissions. This concentration suggests that a few large facilities can significantly increase the per-capita emissions burden in smaller counties, reinforcing the idea that emissions are not evenly distributed across sectors or regions.

*TODO (Veer): Write 2-3 sentences interpreting which industries drive the burden in high per-capita counties.*

---
## 7. Conclusion

*TODO (All): Summarize the key findings:*
- *How does the per-capita picture differ from the total emissions picture?*
- *What role does income play in who bears the emissions burden?*
- *Which sectors are the biggest contributors in the hardest-hit counties?*
- *What are the limitations of this analysis?*